### Installation

In [ ]:
0%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We now use the `Qwen-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Qwen-3 renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3",
)

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for tokens, tags in zip(examples["tokens"], examples["ner_tags"]):
        if tokens[0] == "RT":
            tokens = tokens[3:]
            tags = tags[3:]
        input = " ".join(tokens)
        output = ""
        in_mention = False
        for token, tag_int in zip(tokens, tags):
            tag = tag_int2str(tag_int)
            if tag == "O":
                if in_mention:
                    output += f">> {token} "
                    in_mention = False
                else:
                    output += f"{token} "
            elif tag.startswith("B"):
                if in_mention:
                    output += f">> <<{tag[2:]}: {token} "
                else:
                    output += f"<<{tag[2:]}: {token} "
                    in_mention = True
            else:
                output += f"{token} "
        if in_mention:
            output += ">>"
        convo = create_conversation(input, output.rstrip())
        texts.append(tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False))
    return { "text" : texts, }

In [ ]:
system_message = """You are an expert annotator.
Find named entities in the input and enclose them with << and >>.
Add the entity type (PER:, LOC:, ORG:, or MISC:) after each <<.
Return only the input sentence with the added labels."""

def create_conversation(input, response):
  return [
      {"role": "system", "content": system_message},
      {"role": "user", "content": input},
      {"role": "assistant", "content": response}
    ]

from datasets import load_dataset
train_data = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = "train")
tag_int2str = train_data.features["ner_tags"].feature.int2str
train_data = train_data.map(formatting_prompts_func, batched = True)

In [ ]:
train_data[100]['text']

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

for split in ["dev", "test"]:
    eval_data = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = split)
    with open(f"qwen-{split}-pred.txt", "w", encoding="utf-8") as file:
        for row in eval_data:
            tokens = row["tokens"][3:] if row["tokens"][0] == "RT" else row["tokens"]
            messages = [
                {"role": "system", "content": system_message},
                {"role": "user", "content": " ".join(tokens)},
            ]
            input = tokenizer.apply_chat_template(
                messages,
                tokenize = True,
                add_generation_prompt = True, # Must add for generation
                return_tensors = "pt",
            ).to("cuda")
            output = model.generate(input_ids = input, use_cache = True)
            output = tokenizer.batch_decode(output[:, input.shape[1]:], skip_special_tokens = True)
            print(output)
            file.write("\n".join(output) + "\n")

In [ ]:
def convert_to_inline_format(examples):
    texts = []
    for tokens, tags in zip(examples["tokens"], examples["ner_tags"]):
        if tokens[0] == "RT":
            tokens = tokens[3:]
            tags = tags[3:]
        output = ""
        in_mention = False
        for token, tag_int in zip(tokens, tags):
            tag = tag_int2str(tag_int)
            if tag == "O":
                if in_mention:
                    output += f">> {token} "
                    in_mention = False
                else:
                    output += f"{token} "
            elif tag.startswith("B"):
                if in_mention:
                    output += f">> <<{tag[2:]}: {token} "
                else:
                    output += f"<<{tag[2:]}: {token} "
                    in_mention = True
            else:
                output += f"{token} "
        if in_mention:
            output += ">>"
        texts.append(output.rstrip())
    return { "text" : texts, }

for split in ["dev", "test"]:
    eval_data = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = split)
    tag_int2str = eval_data.features["ner_tags"].feature.int2str
    eval_data = eval_data.map(convert_to_inline_format, batched = True,)
    with open(f"qwen-{split}-true.txt", "w", encoding="utf-8") as file:
        for sentence in eval_data:
            file.write(sentence["text"] + "\n")

In [ ]:
import re

for split in ["dev", "test"]:
    tp = 0
    fp = 0
    fn = 0
    wt = 0
    with open(f"qwen-{split}-true.txt", "r", encoding="utf-8") as file_true, \
         open(f"qwen-{split}-pred.txt", "r", encoding="utf-8") as file_pred, \
         open(f"qwen-{split}-false-positives.txt", "w", encoding="utf-8") as file_fp:
        for line_num, (true, pred) in enumerate(zip(file_true, file_pred), start=1):
            true_mentions = re.findall("<<(.*?)>>", true)
            true_mentions_dict = {}
            for mention in true_mentions:
                entity_type, mention = mention.split(": ", 1)
                true_mentions_dict[mention] = entity_type

            pred_mentions = re.findall("<<(.*?)>>", pred)
            pred_mentions_dict = {}
            for mention in pred_mentions:
                try:
                    entity_type, mention = mention.split(": ", 1)
                    pred_mentions_dict[mention] = entity_type
                except:
                    print("Invalid mention: " + mention)

            for mention, true_type in true_mentions_dict.items():
                pred_type = pred_mentions_dict.get(mention)
                if pred_type == true_type:
                    tp += 1
                elif pred_type is None:
                    fn += 1
                else:   # partial credit for wrong type
                    tp += 0.5
                    fn += 0.5
                    fp += 0.5
                    wt += 1
            for mention in pred_mentions_dict:
                if true_mentions_dict.get(mention) is None:
                    fp += 1
                    file_fp.write(f"{line_num} {mention}\n")

    print(f"***{split}***")
    print(tp, fp, fn, wt)
    print(f"Precision: {tp / (tp + fp)}")
    print(f"Recall: {tp / (tp + fn)}")

# Round 2

## Setup + Data Prep

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3",
)

In [ ]:
system_message = """You are an expert annotator tasked with labeling a word/phrase from a sentence.
If the word/phrase is a named entity, return its type (PER, LOC, ORG, or MISC).
If the word/phrase is not a named entity, return O."""

def create_conversation(input, response):
  return [
      {"role": "system", "content": system_message},
      {"role": "user", "content": input},
      {"role": "assistant", "content": response}
    ]

from datasets import load_dataset
dev_split = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = "dev")
tag_int2str = dev_split.features["ner_tags"].feature.int2str

data_for_training = []

for tokens, tags in zip(dev_split["tokens"], dev_split["ner_tags"]):
    if tokens[0] == "RT":
        tokens = tokens[3:]
        tags = tags[3:]
    entities = []
    curr_mention = ""
    curr_tag = ""
    for token, tag_int in zip(tokens, tags):
        tag = tag_int2str(tag_int)
        if tag.startswith("I"):
            curr_mention += f"{token} "
        else:
            if curr_mention:
                entities.append((curr_mention, curr_tag))
                curr_mention = []
            if tag.startswith("B"):
                curr_mention = f"{token} "
                curr_tag = tag[2:]
    if curr_mention:
        entities.append((curr_mention, curr_tag))
    sentence = " ".join(tokens)
    for entity in entities:
        input = f'Sentence: "{sentence}". The type of "{entity[0]}" is:'
        convo = create_conversation(input, entity[1])
        data_for_training.append(tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False))

In [ ]:
with open("qwen-dev-false-positives.txt", "r", encoding="utf-8") as file_fp:
    for line in file_fp:
        line_num, mention = line.split(" ", 1)
        sentence_tokens = dev_split["tokens"][int(line_num)-1]
        if sentence_tokens[0] == "RT":
            sentence_tokens = sentence_tokens[3:]
        sentence = " ".join(sentence_tokens)
        input = f'Sentence: "{sentence}". The type of "{mention}" is:'
        convo = create_conversation(input, "O")
        data_for_training.append(tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False))

In [ ]:
from datasets import Dataset
train_dataset = Dataset.from_dict({"text": data_for_training})

In [ ]:
print(train_dataset[0]["text"])

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
trainer_stats = trainer.train()

## Eval

In [ ]:
import csv
test_split = load_dataset("bltlab/open-ner-standardized", "Tweebank_eng", split = "test")
tag_int2str = test_split.features["ner_tags"].feature.int2str
test_data = []

for tokens, tags in zip(test_split["tokens"], test_split["ner_tags"]):
    if tokens[0] == "RT":
        tokens = tokens[3:]
        tags = tags[3:]
    entities = []
    curr_mention = ""
    curr_tag = ""
    for token, tag_int in zip(tokens, tags):
        tag = tag_int2str(tag_int)
        if tag.startswith("I"):
            curr_mention += f"{token} "
        else:
            if curr_mention:
                entities.append((curr_mention, curr_tag))
                curr_mention = []
            if tag.startswith("B"):
                curr_mention = f"{token} "
                curr_tag = tag[2:]
    if curr_mention:
        entities.append((curr_mention, curr_tag))
    sentence = " ".join(tokens)
    for entity in entities:
        test_data.append((sentence, entity[0], entity[1]))

In [ ]:
with open("qwen-test-false-positives.txt", "r", encoding="utf-8") as file_fp:
    for line in file_fp:
        line_num, mention = line.split(" ", 1)
        sentence_tokens = test_split["tokens"][int(line_num)-1]
        if sentence_tokens[0] == "RT":
            sentence_tokens = sentence_tokens[3:]
        sentence = " ".join(sentence_tokens)
        test_data.append((sentence, mention, "O"))

In [ ]:
import random
random.seed(0)
random.shuffle(test_data)

with open("qwen-test-true.csv", "w", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Sentence", "Mention", "Type"])
    writer.writerows(test_data)

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

with open("qwen-test-pred-r2.csv", "w", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Sentence", "Mention", "Type"])
    for row in test_data:
        user_input = f'Sentence: "{row[0]}". The type of "{row[1]}" is:'
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_input},
        ]
        input = tokenizer.apply_chat_template(
            messages,
            tokenize = True,
            add_generation_prompt = True, # Must add for generation
            enable_thinking = False,
            return_tensors = "pt",
        ).to("cuda")
        output = model.generate(input_ids = input, use_cache = True)
        output = tokenizer.batch_decode(output[:, input.shape[1]:], skip_special_tokens = True)
        print(output)
        writer.writerow([row[0], row[1], output[0]])

In [ ]:
import re
tp = 0
fp = 0
fn = 0
wt = 0
with open("qwen-test-true.csv", "r", encoding="utf-8") as file_true, \
     open("qwen-test-pred-r2.csv", "r", encoding="utf-8") as file_pred:
    reader_true = csv.reader(file_true)
    reader_pred = csv.reader(file_pred)
    next(reader_true)   # skip header row
    next(reader_pred)
    for true, pred in zip(reader_true, reader_pred):
        if true[2] == pred[2]:
            if true[2] != "O":
                tp += 1
        elif true[2] == "O":
            fp += 1
        elif pred[2] == "O":
            fn += 1
        else:   # partial credit for wrong type
            tp += 0.5
            fn += 0.5
            fp += 0.5
            wt += 1

print(tp, fp, fn, wt)
print(f"Precision: {tp / (tp + fp)}")
print(f"Recall: {tp / (tp + fn)}")